In [1]:
import sys, os
notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
CLOTHILDE_ROOT = parent_dir
EXPORT_DIR = CLOTHILDE_ROOT + "/exported_test_gripper_temp" # By keeping everything float64
# os.makedirs(EXPORT_DIR, exist_ok=True)
export = False

from implementation.Cloth_speed import Cloth
from implementation.utils import createRectangularMesh
import numpy as np

In [2]:
from implementation.Gripper import (
        SimulateGripper,
        quat_from_axis_angle,
        quat_to_rotmat,
        quat_transform_points,
        quat_normalize,
        quat_from_rotvec,
        quat_rotate_vector,
        quat_mul
    )

In [3]:
def read_gripper_poses(export_dir):
    gripper_poses = np.load(os.path.join(export_dir, "gripper_poses_Franka.npz"))
    times = gripper_poses["t"]
    p_gripper = gripper_poses["position"]
    q_gripper = gripper_poses["quaternion"]
    jaw_status = gripper_poses["jaw_status"].astype(int)
    return p_gripper, q_gripper, jaw_status, times

In [4]:
p_gripper, q_gripper, jaw_status, times = read_gripper_poses(EXPORT_DIR)
print(f'gripper quaternion: {q_gripper[100]}')

gripper quaternion: [4.32978028e-17 7.07106781e-01 7.07106781e-01 4.32978028e-17]


In [5]:
quat_normalize(quat_from_axis_angle([0, 1, 0], np.pi/2))

array([0.70710678, 0.        , 0.70710678, 0.        ])

In [7]:
quat_from_axis_angle([0, 1, 0], np.pi)

array([6.123234e-17, 0.000000e+00, 1.000000e+00, 0.000000e+00])

In [28]:
import numpy as np

def quat_to_axis_angle(q):
    """
    Convert quaternion [w, x, y, z] to axis-angle.

    Returns:
        axis: np.ndarray, shape (3,)
        angle: float (radians)
    """
    q = np.asarray(q, dtype=float)
    q = q / np.linalg.norm(q)  # normalize

    w, x, y, z = q

    # Clamp to avoid numerical issues
    w = np.clip(w, -1.0, 1.0)

    angle = 2.0 * np.arccos(w)
    s = np.sqrt(1.0 - w * w)

    if s < 1e-8:
        # Angle is close to 0, axis is arbitrary
        axis = np.array([1.0, 0.0, 0.0])
    else:
        axis = np.array([x, y, z]) / s

    return axis, angle

In [32]:
quat_to_axis_angle([0., 0.70710678, 0.70710678, 0.])

(array([0.70710678, 0.70710678, 0.        ]), np.float64(3.141592653589793))